# Customer Churn Prediction
### Auto-adaptive — works with any churn dataset

This notebook walks through the full churn prediction pipeline:
1. Install dependencies
2. Load data (sample or your own)
3. Explore the data
4. Train the model — auto-selects best algorithm for your data
5. Evaluate results
6. Predict on new customers
7. Inspect feature importances

---
**To use your own data:** replace the sample DataFrame in Section 2 and update `target_col` and `drop_cols`.

## 0. Install Dependencies

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn --quiet
print('Dependencies ready.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import the ChurnModel class from the same folder
# If running standalone, the class definition is also copied below in Section 3
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

print('Imports done.')

## 1. Load Data

Replace the block below with `pd.read_csv('your_file.csv')` and update `TARGET_COL` and `DROP_COLS`.

In [ ]:
# ============================================================
# CONFIG — change these to match your dataset
# ============================================================
TARGET_COL = 'churned'          # column that means 'did this customer churn'
DROP_COLS  = ['customer_id']    # ID columns or anything not useful as features

# ============================================================
# DATA — swap this out for your own DataFrame
# e.g. df = pd.read_csv('telco_churn.csv')
# ============================================================
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    'customer_id':      range(n),
    'tenure_months':    np.random.randint(1, 72, n),
    'monthly_charges':  np.round(np.random.uniform(20, 120, n), 2),
    'total_charges':    np.round(np.random.uniform(100, 8000, n), 2),
    'num_products':     np.random.randint(1, 6, n),
    'support_calls':    np.random.randint(0, 10, n),
    'contract_type':    np.random.choice(['Month-to-Month', 'One Year', 'Two Year'], n),
    'payment_method':   np.random.choice(['Credit Card', 'Bank Transfer', 'Electronic Check'], n),
    'internet_service': np.random.choice(['DSL', 'Fiber', 'No'], n),
    'churned':          np.random.choice([0, 1], n, p=[0.73, 0.27])
})

print(f'Shape: {df.shape}')
print(f'Churn rate: {df[TARGET_COL].mean():.1%}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('=== Dataset Info ===')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')
print(f'Missing values: {df.isnull().sum().sum()}')
print()
print('=== Column Types ===')
print(df.dtypes)
print()
print('=== Target Distribution ===')
print(df[TARGET_COL].value_counts())
print(f'Churn rate: {df[TARGET_COL].mean():.1%}')

In [ ]:
numeric_cols = df.select_dtypes(include=['int64','float64']).columns.drop(TARGET_COL, errors='ignore')
numeric_cols = [c for c in numeric_cols if c not in DROP_COLS]

if len(numeric_cols) > 0:
    fig, axes = plt.subplots(1, len(numeric_cols), figsize=(5 * len(numeric_cols), 4))
    if len(numeric_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, numeric_cols):
        df.groupby(TARGET_COL)[col].plot(kind='kde', ax=ax, legend=True)
        ax.set_title(f'{col} by Churn')
        ax.set_xlabel(col)
    plt.suptitle('Feature Distributions: Churned vs Retained', y=1.02, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

cat_cols = [c for c in df.select_dtypes(include='object').columns if c not in DROP_COLS]
if len(cat_cols) > 0:
    fig, axes = plt.subplots(1, len(cat_cols), figsize=(5 * len(cat_cols), 4))
    if len(cat_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cat_cols):
        churn_rate_by_cat = df.groupby(col)[TARGET_COL].mean().sort_values(ascending=False)
        churn_rate_by_cat.plot(kind='bar', ax=ax, color='#e74c3c', alpha=0.8)
        ax.set_title(f'Churn Rate by {col}')
        ax.set_ylabel('Churn Rate')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
        ax.axhline(df[TARGET_COL].mean(), linestyle='--', color='black', linewidth=1, label='Overall avg')
        ax.legend()
    plt.suptitle('Churn Rate by Categorical Feature', y=1.02, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
numeric_df = df[list(numeric_cols) + [TARGET_COL]].copy()
if len(numeric_df.columns) > 1:
    corr = numeric_df.corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
    ax.set_title('Correlation Matrix')
    plt.tight_layout()
    plt.show()

## 3. ChurnModel Class

This is the same class as `churn_model.py`. Defined here so the notebook is fully self-contained.

In [ ]:
import re
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, ConfusionMatrixDisplay
from xgboost import XGBClassifier


class ChurnModel:
    """
    Auto-adaptive churn prediction model.
    Works with any churn dataset. Set target_col and drop_cols, call fit().
    """

    def __init__(self, target_col='churned', drop_cols=None, test_size=0.2, random_state=42, n_iter=10, cv=5):
        self.target_col   = target_col
        self.drop_cols    = drop_cols or []
        self.test_size    = test_size
        self.random_state = random_state
        self.n_iter       = n_iter
        self.cv           = cv
        self.best_model      = None
        self.best_model_name = None
        self.feature_names_  = None
        self.is_fitted       = False

    def _build_preprocessor(self, X):
        numeric_cols     = X.select_dtypes(include=['int64','float64']).columns.tolist()
        categorical_cols = X.select_dtypes(include=['object','category','bool']).columns.tolist()
        print(f'  Numeric features    ({len(numeric_cols)}): {numeric_cols}')
        print(f'  Categorical features ({len(categorical_cols)}): {categorical_cols}')
        num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
        cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
        transformers = []
        if numeric_cols:     transformers.append(('num', num_pipe, numeric_cols))
        if categorical_cols: transformers.append(('cat', cat_pipe, categorical_cols))
        if not transformers:
            raise ValueError('No numeric or categorical columns found.')
        return ColumnTransformer(transformers=transformers)

    def _get_candidates(self):
        return {
            'XGBoost': (
                XGBClassifier(eval_metric='logloss', random_state=self.random_state),
                {'classifier__n_estimators':[100,200,300],'classifier__max_depth':[3,5,7],
                 'classifier__learning_rate':[0.01,0.1,0.2],'classifier__subsample':[0.7,0.9,1.0]}
            ),
            'Random Forest': (
                RandomForestClassifier(random_state=self.random_state),
                {'classifier__n_estimators':[100,200,300],'classifier__max_depth':[None,5,10,20],
                 'classifier__min_samples_split':[2,5,10]}
            ),
            'Logistic Regression': (
                LogisticRegression(random_state=self.random_state, max_iter=1000),
                {'classifier__C':[0.001,0.01,0.1,1,10,100],'classifier__solver':['lbfgs','saga']}
            ),
        }

    def fit(self, df):
        print('='*50)
        print('  ChurnModel — Training')
        print('='*50)
        if self.target_col not in df.columns:
            raise ValueError(f"target_col '{self.target_col}' not found. Columns: {df.columns.tolist()}")
        cols_to_drop = [self.target_col] + [c for c in self.drop_cols if c in df.columns]
        X = df.drop(columns=cols_to_drop)
        y = df[self.target_col]
        self.feature_names_ = X.columns.tolist()
        churn_rate = y.mean()
        print(f'\nDataset: {len(X)} rows | {len(X.columns)} features | Churn rate: {churn_rate:.1%}')
        if churn_rate < 0.05 or churn_rate > 0.95:
            print(f'  ⚠  Class imbalance detected. Consider SMOTE for better minority-class recall.')
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=self.test_size, random_state=self.random_state, stratify=y)
        print('\nFeature detection:')
        preprocessor = self._build_preprocessor(X_train)
        candidates   = self._get_candidates()
        best_score = -1
        cv_results = {}
        print(f'\nTesting {len(candidates)} algorithms ({self.n_iter} configs x {self.cv}-fold CV each)...')
        for name, (clf, params) in candidates.items():
            pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
            search   = RandomizedSearchCV(pipeline, params, n_iter=self.n_iter, cv=self.cv,
                                          scoring='roc_auc', random_state=self.random_state, n_jobs=-1)
            search.fit(X_train, y_train)
            score = search.best_score_
            cv_results[name] = score
            marker = ' <- best so far' if score > best_score else ''
            print(f'  {name:<25} CV AUC = {score:.4f}{marker}')
            if score > best_score:
                best_score = score
                self.best_model      = search.best_estimator_
                self.best_model_name = name
                self._best_params    = search.best_params_
        print(f'\n✅ Best model: {self.best_model_name} (CV AUC: {best_score:.4f})')
        print(f'   Best params: {self._best_params}')
        y_pred = self.best_model.predict(X_test)
        y_prob = self.best_model.predict_proba(X_test)[:,1]
        print(f'\nTest Set AUC: {roc_auc_score(y_test, y_prob):.4f}')
        print('\nClassification Report:')
        print(classification_report(y_test, y_pred, target_names=['Retained','Churned']))
        self._plot_results(y_test, y_pred, y_prob, cv_results)
        self._X_test = X_test
        self._y_test = y_test
        self._y_prob = y_prob
        self.is_fitted = True
        return self

    def predict(self, df):
        if not self.is_fitted:
            raise RuntimeError('Call fit() before predict().')
        X = df.drop(columns=[c for c in [self.target_col]+self.drop_cols if c in df.columns])
        probs = self.best_model.predict_proba(X)[:,1]
        preds = self.best_model.predict(X)
        return pd.DataFrame({'churn_probability': np.round(probs,4), 'will_churn': preds}, index=df.index)

    def feature_importance(self, top_n=15):
        if not self.is_fitted: raise RuntimeError('Call fit() first.')
        clf_step = self.best_model.named_steps['classifier']
        if not hasattr(clf_step, 'feature_importances_'):
            print(f'{self.best_model_name} does not expose feature importances.'); return
        preprocessor  = self.best_model.named_steps['preprocessor']
        try:    feature_names = preprocessor.get_feature_names_out()
        except: feature_names = [f'feature_{i}' for i in range(len(clf_step.feature_importances_))]
        feature_names = [re.sub(r'^(num__|cat__)', '', n) for n in feature_names]
        importances = clf_step.feature_importances_
        idx = np.argsort(importances)[::-1][:top_n]
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.barh([feature_names[i] for i in reversed(idx)], [importances[i] for i in reversed(idx)], color='#3498db')
        ax.set_xlabel('Importance Score')
        ax.set_title(f'Top {top_n} Feature Importances — {self.best_model_name}')
        plt.tight_layout()
        plt.show()

    def _plot_results(self, y_test, y_pred, y_prob, cv_results):
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'Churn Model Results  |  Best: {self.best_model_name}', fontsize=14, fontweight='bold')
        max_score = max(cv_results.values())
        colors = ['#2ecc71' if v == max_score else '#bdc3c7' for v in cv_results.values()]
        bars = axes[0].barh(list(cv_results.keys()), list(cv_results.values()), color=colors)
        axes[0].set_xlabel('CV ROC-AUC Score')
        axes[0].set_title('Algorithm Comparison')
        axes[0].set_xlim(0, 1.05)
        for bar, v in zip(bars, cv_results.values()):
            axes[0].text(v+0.01, bar.get_y()+bar.get_height()/2, f'{v:.4f}', va='center', fontsize=10)
        ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Retained','Churned'],
                                                ax=axes[1], colorbar=False, cmap='Blues')
        axes[1].set_title('Confusion Matrix (Test Set)')
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        axes[2].plot(fpr, tpr, color='#2ecc71', lw=2, label=f'AUC = {auc:.4f}')
        axes[2].fill_between(fpr, tpr, alpha=0.1, color='#2ecc71')
        axes[2].plot([0,1],[0,1],'k--',lw=1,label='Random baseline')
        axes[2].set_xlabel('False Positive Rate')
        axes[2].set_ylabel('True Positive Rate')
        axes[2].set_title('ROC Curve (Test Set)')
        axes[2].legend(loc='lower right')
        plt.tight_layout()
        plt.show()

print('ChurnModel class ready.')

## 4. Train the Model

In [ ]:
model = ChurnModel(
    target_col=TARGET_COL,
    drop_cols=DROP_COLS
)

model.fit(df)

## 5. Feature Importances

See which features drive churn the most (tree-based models only).

In [ ]:
model.feature_importance(top_n=15)

## 6. Predict on New Customers

Pass a DataFrame with the same feature columns — no label needed.

In [ ]:
new_customers = pd.DataFrame({
    'customer_id':      [9001, 9002, 9003, 9004],
    'tenure_months':    [2, 36, 60, 1],
    'monthly_charges':  [95.0, 45.0, 30.0, 110.0],
    'total_charges':    [190.0, 1620.0, 1800.0, 110.0],
    'num_products':     [1, 3, 5, 1],
    'support_calls':    [8, 1, 0, 9],
    'contract_type':    ['Month-to-Month', 'One Year', 'Two Year', 'Month-to-Month'],
    'payment_method':   ['Electronic Check', 'Credit Card', 'Bank Transfer', 'Electronic Check'],
    'internet_service': ['Fiber', 'DSL', 'DSL', 'Fiber']
})

predictions = model.predict(new_customers)

result = pd.concat([
    new_customers[['customer_id', 'tenure_months', 'monthly_charges', 'contract_type']],
    predictions
], axis=1)

result['risk_tier'] = pd.cut(
    result['churn_probability'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

print('Churn Predictions:')
result

---
## 7. Using Your Own Dataset

```python
# Load any churn CSV
df = pd.read_csv('your_churn_data.csv')

# Set these to match YOUR column names
TARGET_COL = 'Churn'           # or 'is_cancelled', 'subscription_ended', etc.
DROP_COLS  = ['CustomerID']    # ID columns, timestamp columns, etc.

model = ChurnModel(target_col=TARGET_COL, drop_cols=DROP_COLS)
model.fit(df)

# Predict on new rows
predictions = model.predict(new_df)
```

### Public datasets to test with
- [IBM Telco Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn) — `target_col='Churn'`
- [E-Commerce Churn](https://www.kaggle.com/datasets/ankitverma2010/ecommerce-customer-churn-analysis-and-prediction) — `target_col='Churn'`

---
## Coming Next in This Library
- `models/sentiment/` — Sentiment Analysis
- `models/segmentation/` — Customer Segmentation (K-Means)
- `models/fraud/` — Fraud Detection
- `agent/router.py` — LLM Agent that routes business problems to the right model